# PTM rescoring with and without localization

This notebook rescores PTM peptides from a single raw file of an HLA immunopeptidomics dataset
([PXD009738](https://www.ebi.ac.uk/pride/archive/projects/PXD009738), searched with MSFragger) with a PTM-aware Prosit model.

Oktoberfest offers two modes for PTMs:

- **Without localization** (`"ptm_localization": false`): PTMs are rescored at the sites reported by the search engine.
  With `"ptm_fdr_control": true`, unmodified and modified PSMs get a separate FDR estimation (modified PSMs in `results/percolator/mod_fdr`).
- **With localization** (`"ptm_localization": true`): for the PTMs **you choose**, all possible localizations on the residues **you choose**
  are generated and scored, and the best localization per spectrum is selected.

The PTMs to localize are given as a mapping of UNIMOD id to the residues they can be located on, e.g. `{"21": ["S", "T", "Y"]}`
for phosphorylation or `{"21": ["S", "T", "Y"], "7": ["R"]}` for phosphorylation and citrullination.

## 1- Import necessary python packages

In [ ]:
import json
from pathlib import Path

import pandas as pd

from oktoberfest.runner import run_job

## 2- Input files

One raw file (as mzML) and its MSFragger search result (pepXML). The file names (without extension) must match.

In [ ]:
data_dir = Path("/cmnfs/home/w.gabriel/new_oktoberfest/notebook_runs/inputs/hla_one")
spectra = str(data_dir)                 # folder with the mzML file
search_results = str(data_dir / "FL0179b_MON_0267N_even.pepXML")
output_base = Path("./out_ptm_hla")

# PTM-aware Prosit models available on Koina
intensity_model = "Prosit_2024_intensity_PTMs_gl"
irt_model = "Prosit_2024_irt_PTMs_gl"

In [ ]:
def make_config(output_dir: Path, **ptm_options) -> dict:
    """Rescoring config for the HLA file, extended by the given PTM options."""
    return {
        "type": "Rescoring",
        "tag": "",
        "inputs": {
            "search_results": search_results,
            "search_results_type": "MSFragger",
            "spectra": spectra,
            "spectra_type": "mzml",
        },
        "output": str(output_dir.resolve()),  # relative paths are resolved against the config file location
        "models": {"intensity": intensity_model, "irt": irt_model},
        "prediction_server": "koina.wilhelmlab.org:443",
        "ssl": True,
        "numThreads": 4,
        "fdr_estimation_method": "percolator",
        "regressionMethod": "spline",
        "allFeatures": False,
        "massTolerance": 20,
        "unitMassTolerance": "ppm",
        **ptm_options,
    }


def summarize(results_dir: Path, ptm_pattern: str = r"UNIMOD:21\]", fdr: float = 0.01) -> pd.DataFrame:
    """Count target PSMs / peptides below the FDR cutoff for every percolator result folder of a run."""
    rows = []
    for psm_file in sorted(results_dir.rglob("rescore.percolator.psms.txt")):
        psms = pd.read_csv(psm_file, sep="\t")
        psms = psms[psms["q-value"] <= fdr]
        rows.append(
            {
                "result folder": str(psm_file.parent.relative_to(results_dir)),
                "PSMs": len(psms),
                "PTM PSMs": int(psms["peptide"].str.contains(ptm_pattern, regex=True).sum()),
                "PTM peptides": psms.loc[psms["peptide"].str.contains(ptm_pattern, regex=True), "peptide"].nunique(),
            }
        )
    return pd.DataFrame(rows)

## 3- PTM rescoring without localization

PTMs are kept at the sites reported by MSFragger. Modified and unmodified PSMs get a separate FDR estimation.

In [ ]:
output_no_loc = output_base / "without_localization"
config_no_loc = make_config(output_no_loc, ptm_localization=False, ptm_fdr_control=True)

output_no_loc.mkdir(parents=True, exist_ok=True)
with open(output_no_loc / "config.json", "w") as fp:
    json.dump(config_no_loc, fp, indent=2)
config_no_loc

In [ ]:
run_job(output_no_loc / "config.json")

In [ ]:
summary_no_loc = summarize(output_no_loc / "results" / "percolator")
summary_no_loc

## 4- PTM rescoring with localization

Choose the PTMs to localize and the residues each of them can be located on (UNIMOD id -> residues).
Here: phosphorylation (UNIMOD:21) on S, T and Y. Add more entries to localize several PTMs together,
e.g. `{"21": ["S", "T", "Y"], "7": ["R"]}`.

In [ ]:
ptms_to_localize = {"21": ["S", "T", "Y"]}

In [ ]:
output_loc = output_base / "with_localization"
config_loc = make_config(
    output_loc,
    ptm_localization=True,
    ptmLocalizationOptions={"ptms": ptms_to_localize, "neutral_loss": True},
)

output_loc.mkdir(parents=True, exist_ok=True)
with open(output_loc / "config.json", "w") as fp:
    json.dump(config_loc, fp, indent=2)
config_loc

In [ ]:
run_job(output_loc / "config.json")

In [ ]:
summary_loc = summarize(output_loc / "results" / "percolator")
summary_loc

## 5- Compare both modes

In [ ]:
pd.concat({"without localization": summary_no_loc, "with localization": summary_loc}, names=["mode"]).reset_index(level=0)

The localized phosphopeptides (best scoring localization per spectrum, 1% FDR) can be found in the result folder
of the localization run, e.g. `results/percolator/localize_phospho/rescore.percolator.psms.txt` for phosphorylation.

In [ ]:
localized_psm_files = sorted((output_loc / "results" / "percolator").glob("localize_*/rescore.percolator.psms.txt"))
localized = pd.read_csv(localized_psm_files[0], sep="\t") if localized_psm_files else pd.DataFrame()
localized[localized["q-value"] <= 0.01].head(10) if len(localized) else "no localization results found"